In [324]:
from pathlib import Path
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [325]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder exists:", RAW_DIR.exists())
print("Output folder:", PROCESSED_DIR.resolve())



Raw folder exists: True
Output folder: C:\Desktop\ICC_World_Cup_2023\data\processed


In [326]:
matches = pd.read_csv(RAW_DIR / "matches.csv")
deliveries = pd.read_csv(RAW_DIR / "deliveries.csv")
points_table = pd.read_csv(RAW_DIR / "points_table.csv")




In [327]:
baseline = {
    "deliveries_rows": len(deliveries),
    "matches_rows": len(matches),
    "points_table_rows": len(points_table)
}
baseline

{'deliveries_rows': 26119, 'matches_rows': 48, 'points_table_rows': 10}

In [328]:
print("Duplicate delivery rows:", deliveries.duplicated().sum())

duplicate_rows = deliveries[deliveries.duplicated(keep=False)].sort_values(deliveries.columns.tolist())
display(duplicate_rows)

Duplicate delivery rows: 1


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
10383,21,2023/24,2023-10-22,"Himachal Pradesh Cricket Association Stadium, ...",1,9.1,New Zealand,India,DJ Mitchell,R Ravindra,Mohammed Siraj,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10392,21,2023/24,2023-10-22,"Himachal Pradesh Cricket Association Stadium, ...",1,9.1,New Zealand,India,DJ Mitchell,R Ravindra,Mohammed Siraj,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [329]:

#  Exact duplicate rows remove karo
deliveries = deliveries.drop_duplicates()

print("Deliveries after duplication:", deliveries.shape)
print("Remaining duplicates:", deliveries.duplicated().sum())

Deliveries after duplication: (26118, 22)
Remaining duplicates: 0


In [330]:
# column names and text values clean 
def clean_text_columns(df):
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )
    return df

deliveries = clean_text_columns(deliveries)
matches = clean_text_columns(matches)
points_table = clean_text_columns(points_table)

print(deliveries.columns.tolist())
print(matches.columns.tolist())
print(points_table.columns.tolist())



['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']
['season', 'team1', 'team2', 'date', 'match_number', 'venue', 'city', 'toss_winner', 'toss_decision', 'player_of_match', 'umpire1', 'umpire2', 'reserve_umpire', 'match_referee', 'winner', 'winner_runs', 'winner_wickets', 'match_type']
['ranking', 'team', 'matches', 'won', 'lost', 'tie', 'no_results', 'points', 'net_run_rate', 'for', 'against']


In [331]:

# Reload the original matches dataset
matches_raw = pd.read_csv(RAW_DIR / "matches.csv")

print("Original date examples:")
display(matches_raw[["date", "team1", "team2"]].head(10))

print("\nOriginal date data type:")
print(matches_raw["date"].dtype)

print("\nOriginal missing dates:")
print(matches_raw["date"].isna().sum())

# Check which dates failed after the current conversion
raw_dates = matches_raw["date"].astype("string").str.strip()

parsed_dates = pd.to_datetime(
    raw_dates,
    format="mixed",
    errors="coerce",
    dayfirst=True
)

print("\nMissing dates with mixed-format parsing:")
print(parsed_dates.isna().sum())

print("\nDates that could not be parsed:")
display(matches_raw.loc[
    parsed_dates.isna(),
    ["date", "team1", "team2"]
])

Original date examples:


,date,team1,team2
0,2023/10/05,England,New Zealand
1,2023/10/06,Pakistan,Netherlands
2,2023/10/07,Afghanistan,Bangladesh
3,2023/10/07,South Africa,Sri Lanka
4,2023/10/08,Australia,India
5,2023/10/09,New Zealand,Netherlands
6,2023/10/10,England,Bangladesh
7,2023/10/10,Sri Lanka,Pakistan
8,2023/10/11,Afghanistan,India
9,2023/10/12,South Africa,Australia



Original date data type:
object

Original missing dates:
0

Missing dates with mixed-format parsing:
0

Dates that could not be parsed:


,date,team1,team2


In [332]:
print("Dataset shape:", deliveries.shape)
print("\nColumn names:")
print(deliveries.columns.tolist())

display(deliveries.head())

Dataset shape: (26118, 22)

Column names:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
0,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.1,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.2,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,6,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.3,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.4,England,New Zealand,DJ Malan,JM Bairstow,TA Boult,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.5,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [333]:
deliveries.columns = (
    deliveries.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(deliveries.columns.tolist())

['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']


In [334]:
text_cols = deliveries.select_dtypes(
    include=["object", "string"]
).columns

for col in text_cols:
    deliveries[col] = deliveries[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )
    
deliveries = deliveries.replace(r"^\s*$", np.nan, regex=True)

print("Text cleaning complete.")

Text cleaning complete.


In [335]:
# date datatype
deliveries["start_date"] = pd.to_datetime(
    deliveries["start_date"],
    errors="coerce"
)

print("Date datatype:", deliveries["start_date"].dtype)
print("Missing/invalid dates:", deliveries["start_date"].isna().sum())

display(deliveries[["start_date"]].head())

Date datatype: datetime64[ns]
Missing/invalid dates: 0


,start_date
0,2023-10-05
1,2023-10-05
2,2023-10-05
3,2023-10-05
4,2023-10-05


In [336]:

# Correct date conversion

deliveries["start_date"] = pd.to_datetime(
    deliveries["start_date"],
    errors="coerce"
)

matches["date"] = pd.to_datetime(
    matches["date"],
    format="%Y/%m/%d",
    errors="coerce"
)

print("Missing delivery dates:", deliveries["start_date"].isna().sum())
print("Missing match dates:", matches["date"].isna().sum())

display(matches[["date", "team1", "team2"]].head(10))

Missing delivery dates: 0
Missing match dates: 0


,date,team1,team2
0,2023-10-05,England,New Zealand
1,2023-10-06,Pakistan,Netherlands
2,2023-10-07,Afghanistan,Bangladesh
3,2023-10-07,South Africa,Sri Lanka
4,2023-10-08,Australia,India
5,2023-10-09,New Zealand,Netherlands
6,2023-10-10,England,Bangladesh
7,2023-10-10,Sri Lanka,Pakistan
8,2023-10-11,Afghanistan,India
9,2023-10-12,South Africa,Australia


In [337]:
# Extras columns ko clean
extra_columns = [
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty"
]

print("Missing values before filling:")
display(deliveries[extra_columns].isna().sum())

# Verify that non-missing values are numeric
for col in extra_columns:
    deliveries[col] = pd.to_numeric(
        deliveries[col],
        errors="coerce"
    )

# Missing extra subtypes mean no such extra,
# provided this matches the dataset's conventions.
deliveries[extra_columns] = (
    deliveries[extra_columns].fillna(0)
)

print("Missing values after filling:")
display(deliveries[extra_columns].isna().sum())

Missing values before filling:


wides      25453
noballs    26062
byes       26080
legbyes    25925
penalty    26117
dtype: int64

Missing values after filling:


wides      0
noballs    0
byes       0
legbyes    0
penalty    0
dtype: int64

In [338]:

# Delivery numeric columns
delivery_numeric = [
    "match_id", "innings", "ball", "runs_off_bat", "extras"
]

# Match numeric columns
match_numeric = [
    "match_number", "winner_runs", "winner_wickets"
]

# Points table numeric columns
points_numeric = [
    "ranking", "matches", "won", "lost", "tie",
    "no_results", "points", "net_run_rate"
]

def convert_numeric(df, columns, dataset_name):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            print(f"{dataset_name}: Column missing -> {col}")

convert_numeric(deliveries, delivery_numeric, "Deliveries")
convert_numeric(matches, match_numeric, "Matches")
convert_numeric(points_table, points_numeric, "Points table")

print("Delivery numeric columns:")
display(deliveries[[c for c in delivery_numeric if c in deliveries.columns]].describe().T)

print("Match result columns:")
display(matches[[c for c in ["winner_runs", "winner_wickets"] if c in matches.columns]].describe().T)

print("Points table:")
display(points_table.head())

Delivery numeric columns:


,count,mean,std,min,25%,50%,75%,max
match_id,26118.0,24.586645,13.996963,1.0,12.0,24.0,37.0,48.0
innings,26118.0,1.448886,0.497390,1.0,1.0,1.0,2.0,2.0
ball,26118.0,22.648009,13.742183,0.1,10.6,22.1,34.1,49.9
runs_off_bat,26118.0,0.896968,1.399978,0.0,0.0,0.0,1.0,6.0
extras,26118.0,0.046405,0.293629,0.0,0.0,0.0,0.0,5.0


Match result columns:


,count,mean,std,min,25%,50%,75%,max
winner_runs,24.0,125.916667,82.242968,5.0,69.75,101.0,160.00,309.0
winner_wickets,24.0,5.916667,2.019829,1.0,5.00,6.0,7.25,9.0


Points table:


,ranking,team,matches,won,lost,tie,no_results,points,net_run_rate,for,against
0,1,India,9,9,0,0,0,18,2.570,2523/396.2,1708/450.0
1,2,South Africa,9,7,2,0,0,14,1.261,2685/437.5,2158/443.0
2,3,Australia,9,7,2,0,0,14,0.841,2631/426.5,2349/441.2
3,4,New Zealand,9,5,4,0,0,10,0.743,2315/378.0,2279/423.3
4,5,Pakistan,9,4,5,0,0,8,-0.199,2328/406.2,2385/402.2


In [339]:
print("Negative runs off bat:")
print((deliveries["runs_off_bat"] < 0).sum())

print("\nInvalid innings values:")
print((~deliveries["innings"].isin([1, 2])).sum())

print("\nMissing match IDs:")
print(deliveries["match_id"].isna().sum())

print("\nUnique delivery match IDs:")
print(deliveries["match_id"].nunique())

print("\nUnique match records:")
print(matches["match_number"].nunique())

print("\nMissing match winners:")
print(matches["winner"].isna().sum())

print("\nPoints table matches consistency:")
display(
    points_table[
        ["team", "matches", "won", "lost", "tie", "no_results", "points"]
    ]
)

Negative runs off bat:
0

Invalid innings values:
0

Missing match IDs:
0

Unique delivery match IDs:
48

Unique match records:
48

Missing match winners:
0

Points table matches consistency:


,team,matches,won,lost,tie,no_results,points
0,India,9,9,0,0,0,18
1,South Africa,9,7,2,0,0,14
2,Australia,9,7,2,0,0,14
3,New Zealand,9,5,4,0,0,10
4,Pakistan,9,4,5,0,0,8
5,Afghanistan,9,4,5,0,0,8
6,England,9,3,6,0,0,6
7,Bangladesh,9,2,7,0,0,4
8,Sri Lanka,9,2,7,0,0,4
9,Netherlands,9,2,7,0,0,4


In [340]:

# reserve_umpire 
matches[matches["reserve_umpire"].isnull()]

matches["reserve_umpire"] = matches["reserve_umpire"].fillna("Not Available")
matches["reserve_umpire"].isna().sum()

np.int64(0)

In [341]:
matches[
    ["winner", "winner_runs", "winner_wickets"]
].head(15)

,winner,winner_runs,winner_wickets
0,New Zealand,NaN,9.0
1,Pakistan,81.0,NaN
2,Bangladesh,NaN,6.0
3,South Africa,102.0,NaN
4,India,NaN,6.0
5,New Zealand,99.0,NaN
6,England,137.0,NaN
7,Pakistan,NaN,6.0
8,India,NaN,8.0
9,South Africa,134.0,NaN


In [342]:
matches["win_type"] = np.where(
    matches["winner_runs"].notna(),
    "Runs",
    np.where(
        matches["winner_wickets"].notna(),
        "Wickets",
        "Unkhown"
    )
    
)

matches[
    ["winner", "winner_runs", "winner_wickets", "win_type"]
].head(15)

,winner,winner_runs,winner_wickets,win_type
0,New Zealand,NaN,9.0,Wickets
1,Pakistan,81.0,NaN,Runs
2,Bangladesh,NaN,6.0,Wickets
3,South Africa,102.0,NaN,Runs
4,India,NaN,6.0,Wickets
5,New Zealand,99.0,NaN,Runs
6,England,137.0,NaN,Runs
7,Pakistan,NaN,6.0,Wickets
8,India,NaN,8.0,Wickets
9,South Africa,134.0,NaN,Runs


In [343]:
print(matches["win_type"].value_counts(dropna=False))

win_type
Wickets    24
Runs       24
Name: count, dtype: int64


In [344]:
matches[
    ["winner", "winner_runs", "winner_wickets", "win_type"]
].head(5)

,winner,winner_runs,winner_wickets,win_type
0,New Zealand,NaN,9.0,Wickets
1,Pakistan,81.0,NaN,Runs
2,Bangladesh,NaN,6.0,Wickets
3,South Africa,102.0,NaN,Runs
4,India,NaN,6.0,Wickets


In [345]:
matches = matches.drop(
    columns=["winner_runs", "winner_wickets"]
)
matches.columns

Index(['season', 'team1', 'team2', 'date', 'match_number', 'venue', 'city',
       'toss_winner', 'toss_decision', 'player_of_match', 'umpire1', 'umpire2',
       'reserve_umpire', 'match_referee', 'winner', 'match_type', 'win_type'],
      dtype='object')

In [346]:
matches["date"] = pd.to_datetime(
    matches["date"],
    format="%Y-%m-%d",
    errors="coerce"
)
print(matches["date"].head(10))
print("Missing dates:", matches["date"].isna().sum())

0   2023-10-05
1   2023-10-06
2   2023-10-07
3   2023-10-07
4   2023-10-08
5   2023-10-09
6   2023-10-10
7   2023-10-10
8   2023-10-11
9   2023-10-12
Name: date, dtype: datetime64[ns]
Missing dates: 0


In [347]:
# Wicket type
deliveries["wicket_type"] = (
    deliveries["wicket_type"].fillna("No Wicket")
)
deliveries["wicket_type"].value_counts()

wicket_type
No Wicket            25389
caught                 448
bowled                 141
lbw                     76
run out                 38
caught and bowled       14
stumped                 11
retired hurt             1
Name: count, dtype: int64

In [348]:
deliveries["wicket_type"].isna().sum()

np.int64(0)

In [349]:
# Player Dismissal
deliveries["player_dismissed"] = (
    deliveries["player_dismissed"].fillna("Not Out")
)
print(deliveries["player_dismissed"].isnull().sum())
print(deliveries["player_dismissed"].value_counts(dropna=False).head(10))

0
player_dismissed
Not Out               25389
RG Sharma                11
DA Warner                11
Q de Kock                10
DJ Malan                  9
Rahmanullah Gurbaz        9
MP O'Dowd                 9
BFW de Leede              9
Liton Das                 9
CN Ackermann              9
Name: count, dtype: int64


In [350]:
deliveries = deliveries.drop(
    columns=["other_wicket_type", "other_player_dismissed"], errors="ignore"
)


In [351]:
deliveries["start_date"] = pd.to_datetime(
    deliveries["start_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [352]:
expected_columns = [
    "season",
    "team1",
    "team2",
    "date",
    "match_number",
    "venue",
    "city",
    "toss_winner",
    "toss_decision",
    "player_of_match",
    "umpire1",
    "umpire2",
    "reserve_umpire",
    "match_referee",
    "winner",
    "match_type",
    "win_type"
    
]

# Sirf tab reorder karo jab saare expected columns available hon
missing_columns = [
    col for col in expected_columns
    if col not in matches.columns
]

if not missing_columns:
    matches = matches[expected_columns].copy()
    print("Column order corrected!")
else:
    print("Missing columns:", missing_columns)
    print("Actual columns:", matches.columns.tolist())

Column order corrected!


In [353]:
matches["date"] = pd.to_datetime(
    matches["date"],
    errors="coerce"
)

print(matches[["date", "venue"]].head())
print("Invalid dates:", matches["date"].isna().sum())

        date                                         venue
0 2023-10-05                         Narendra Modi Stadium
1 2023-10-06            Rajiv Gandhi International Stadium
2 2023-10-07  Himachal Pradesh Cricket Association Stadium
3 2023-10-07                          Arun Jaitley Stadium
4 2023-10-08                        MA Chidambaram Stadium
Invalid dates: 0


In [354]:
print("Shape:", matches.shape)
print("Columns:", matches.columns.tolist())

display(matches.head())

Shape: (48, 17)
Columns: ['season', 'team1', 'team2', 'date', 'match_number', 'venue', 'city', 'toss_winner', 'toss_decision', 'player_of_match', 'umpire1', 'umpire2', 'reserve_umpire', 'match_referee', 'winner', 'match_type', 'win_type']


,season,team1,team2,date,match_number,venue,city,toss_winner,toss_decision,player_of_match,umpire1,umpire2,reserve_umpire,match_referee,winner,match_type,win_type
0,2023/24,England,New Zealand,2023-10-05,1,Narendra Modi Stadium,Ahmedabad,New Zealand,field,R Ravindra,HDPK Dharmasena,Nitin Menon,Sharfuddoula,J Srinath,New Zealand,Group,Wickets
1,2023/24,Pakistan,Netherlands,2023-10-06,2,Rajiv Gandhi International Stadium,Hyderabad,Netherlands,field,Saud Shakeel,AT Holdstock,CM Brown,PR Reiffel,JJ Crowe,Pakistan,Group,Runs
2,2023/24,Afghanistan,Bangladesh,2023-10-07,3,Himachal Pradesh Cricket Association Stadium,Dharamsala,Bangladesh,field,Mehedi Hasan Miraz,JS Wilson,HDPK Dharmasena,M Erasmus,AJ Pycroft,Bangladesh,Group,Wickets
3,2023/24,South Africa,Sri Lanka,2023-10-07,4,Arun Jaitley Stadium,Delhi,Sri Lanka,field,AK Markram,RK Illingworth,Sharfuddoula,AG Wharf,J Srinath,South Africa,Group,Runs
4,2023/24,Australia,India,2023-10-08,5,MA Chidambaram Stadium,Chennai,Australia,bat,KL Rahul,CB Gaffaney,RA Kettleborough,AT Holdstock,RB Richardson,India,Group,Wickets


In [355]:
text_cols = matches.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    matches[col] = matches[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

matches = matches.replace(r"^\s*$", np.nan, regex=True)
matches.head()

,season,team1,team2,date,match_number,venue,city,toss_winner,toss_decision,player_of_match,umpire1,umpire2,reserve_umpire,match_referee,winner,match_type,win_type
0,2023/24,England,New Zealand,2023-10-05,1,Narendra Modi Stadium,Ahmedabad,New Zealand,field,R Ravindra,HDPK Dharmasena,Nitin Menon,Sharfuddoula,J Srinath,New Zealand,Group,Wickets
1,2023/24,Pakistan,Netherlands,2023-10-06,2,Rajiv Gandhi International Stadium,Hyderabad,Netherlands,field,Saud Shakeel,AT Holdstock,CM Brown,PR Reiffel,JJ Crowe,Pakistan,Group,Runs
2,2023/24,Afghanistan,Bangladesh,2023-10-07,3,Himachal Pradesh Cricket Association Stadium,Dharamsala,Bangladesh,field,Mehedi Hasan Miraz,JS Wilson,HDPK Dharmasena,M Erasmus,AJ Pycroft,Bangladesh,Group,Wickets
3,2023/24,South Africa,Sri Lanka,2023-10-07,4,Arun Jaitley Stadium,Delhi,Sri Lanka,field,AK Markram,RK Illingworth,Sharfuddoula,AG Wharf,J Srinath,South Africa,Group,Runs
4,2023/24,Australia,India,2023-10-08,5,MA Chidambaram Stadium,Chennai,Australia,bat,KL Rahul,CB Gaffaney,RA Kettleborough,AT Holdstock,RB Richardson,India,Group,Wickets


In [356]:
matches["date"] = pd.to_datetime(
    matches["date"],
    
    errors="coerce"
)

print("Invalid dates:", matches["date"].isna().sum())
display(matches[["date"]].head())

Invalid dates: 0


,date
0,2023-10-05
1,2023-10-06
2,2023-10-07
3,2023-10-07
4,2023-10-08


In [357]:
deliveries.columns = (
    deliveries.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

text_cols = deliveries.select_dtypes(
    include=["object", "string"]
).columns

for col in text_cols:
    deliveries[col] = deliveries[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

deliveries = deliveries.replace(r"^\s*$", np.nan, regex=True)

print(deliveries.columns.tolist())

['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed']


In [358]:
print("Duplicate rows:", deliveries.duplicated().sum())

Duplicate rows: 0


In [359]:
deliveries = deliveries.drop_duplicates().copy()

print("Shape after duplicate removal:", deliveries.shape)

Shape after duplicate removal: (26118, 20)


In [360]:
if "start_date" in deliveries.columns:
    deliveries["start_date"] = pd.to_datetime(
        deliveries["start_date"],
        errors="coerce"
    )

    print("Date datatype:", deliveries["start_date"].dtype)
    print("Missing/invalid dates:", deliveries["start_date"].isna().sum())

Date datatype: datetime64[ns]
Missing/invalid dates: 0


In [361]:
integer_cols = [
    "match_id",
    "innings",
    "runs_off_bat",
    "extras",
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty"
]

for col in integer_cols:
    if col in deliveries.columns:
        deliveries[col] = pd.to_numeric(
            deliveries[col],
            errors="coerce"
        ).astype("Int64")

if "ball" in deliveries.columns:
    deliveries["ball"] = pd.to_numeric(
        deliveries["ball"],
        errors="coerce"
    )

print(deliveries.dtypes)


match_id                     Int64
season                      object
start_date          datetime64[ns]
venue                       object
innings                      Int64
ball                       float64
batting_team                object
bowling_team                object
striker                     object
non_striker                 object
bowler                      object
runs_off_bat                 Int64
extras                       Int64
wides                        Int64
noballs                      Int64
byes                         Int64
legbyes                      Int64
penalty                      Int64
wicket_type                 object
player_dismissed            object
dtype: object


In [362]:
missing_report = pd.DataFrame({
    "Missing_Count": deliveries.isnull().sum(),
    "Missing_Percentage": (
        deliveries.isnull().mean() * 100
    ).round(2)
})

display(missing_report)

,Missing_Count,Missing_Percentage
match_id,0,0.0
season,0,0.0
start_date,0,0.0
venue,0,0.0
innings,0,0.0
ball,0,0.0
batting_team,0,0.0
bowling_team,0,0.0
striker,0,0.0
non_striker,0,0.0


In [363]:
if "wicket_type" in deliveries.columns:
    deliveries["wicket_type"] = deliveries["wicket_type"].fillna(
        "No Wicket"
    )

In [364]:
print("Final shape:", deliveries.shape)
print("Exact duplicate rows:", deliveries.duplicated().sum())

if "match_id" in deliveries.columns:
    print("Unique matches:", deliveries["match_id"].nunique())

if "start_date" in deliveries.columns:
    print(
        "Date range:",
        deliveries["start_date"].min(),
        "to",
        deliveries["start_date"].max()
    )

display(deliveries.head())

Final shape: (26118, 20)
Exact duplicate rows: 0
Unique matches: 48
Date range: 2023-10-05 00:00:00 to 2023-11-19 00:00:00


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed
0,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.1,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,0,0,0,0,0,0,0,No Wicket,Not Out
1,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.2,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,6,0,0,0,0,0,0,No Wicket,Not Out
2,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.3,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,1,0,0,0,0,0,0,No Wicket,Not Out
3,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.4,England,New Zealand,DJ Malan,JM Bairstow,TA Boult,1,0,0,0,0,0,0,No Wicket,Not Out
4,1,2023/24,2023-10-05,"Narendra Modi Stadium, Ahmedabad",1,0.5,England,New Zealand,JM Bairstow,DJ Malan,TA Boult,4,0,0,0,0,0,0,No Wicket,Not Out


In [365]:
# cleaned datasets save 
deliveries.to_csv(
    PROCESSED_DIR / "deliveries_cleaned.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S",
    encoding="utf-8",
    na_rep=""
)

matches.to_csv(
    PROCESSED_DIR / "matches_cleaned.csv",
    index=False,
    date_format="%Y-%m-%d"
    
)

points_table.to_csv(
    PROCESSED_DIR / "points_table_cleaned.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

print("Saved files:")
for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(file.name)

All cleaned datasets saved successfully!
Saved files:
deliveries_cleaned.csv
matches_cleaned.csv
points_table_cleaned.csv


In [366]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Deliveries dataset
deliveries.to_csv(
    PROCESSED_DIR / "deliveries_cleaned.csv",
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8",
    na_rep=""
)

# 2. Matches dataset
matches.to_csv(
    PROCESSED_DIR / "matches_cleaned.csv",
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8",
    na_rep=""
)

# 3. Points table dataset
points_table.to_csv(
    PROCESSED_DIR / "points_table_cleaned.csv",
    index=False,
    encoding="utf-8",
    na_rep=""
)

print("All cleaned datasets saved successfully!")

# Show saved files and their shapes
for filename in [
    "deliveries_cleaned.csv",
    "matches_cleaned.csv",
    "points_table_cleaned.csv"
]:
    file_path = PROCESSED_DIR / filename
    saved_df = pd.read_csv(file_path)

    print(f"{filename}: {saved_df.shape}")
    print(f"Location: {file_path.resolve()}")

All cleaned datasets saved successfully!
deliveries_cleaned.csv: (26118, 20)
Location: C:\Desktop\ICC_World_Cup_2023\data\processed\deliveries_cleaned.csv
matches_cleaned.csv: (48, 17)
Location: C:\Desktop\ICC_World_Cup_2023\data\processed\matches_cleaned.csv
points_table_cleaned.csv: (10, 11)
Location: C:\Desktop\ICC_World_Cup_2023\data\processed\points_table_cleaned.csv
